# Stacked Variational Phasor Circuits (VPC) - Regular Stacking
In this notebook, we extend the foundational single-block VPC into a Deep Stacked pattern for machine learning, but this time **without** any non-linear thresholding or normalization between layers. We want to test whether simply allowing the phase values to cascade directly into the next VPC layer's `ShiftGate` can retain original structural coherency without explicitly dragging the state vector's amplitude back to the $N$-Torus manifold.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import phasorflow as pf
from phasorflow.engine.analytic import AnalyticEngine
import torch
import torch.nn as nn
import torch.optim as optim
import math

torch.manual_seed(42)

## 1. Synthetic Data Generation
We will create a synthetic binary classification dataset with $N=12$ features. This mirrors the previous VPC architectures to provide a baseline comparison.

In [2]:
num_samples = 150
num_features = 12 

# Generate random features in range [0, 2pi]
X = torch.empty(num_samples, num_features).uniform_(0, 2*math.pi)

# Create a true non-linear boundary based on the complex phases
# E.g. If the sum of cosine projections is > 0, Class 1, else Class 0
true_signals = torch.sum(torch.cos(X), dim=1)
y = (true_signals > 0).float()

print(f"Dataset generated: {X.shape} features, {y.shape} labels.")

Dataset generated: torch.Size([150, 12]) features, torch.Size([150]) labels.


## 2. Defining the Stacked VPC Structure
We define a modular `create_vpc_block` function.

In [3]:
def create_vpc_block(x_phases, weights_block, block_id=0):
    """
    Builds a single VPC block based on intermediate phases and parameters `weights_block`.
    Weights should be an array of length 24 (12 for Layer 1, 12 for Layer 2).
    """
    pc = pf.PhasorCircuit(num_features, name=f"VPC_Block_{block_id}")
    
    # --- ENCODING ---
    for i in range(num_features):
        pc.shift(i, x_phases[i])
        
    # --- VARIATIONAL LAYER 1 ---
    for i in range(num_features):
        pc.shift(i, weights_block[i])
        
    # Introduce local interference
    for i in range(0, num_features-1, 2):
        pc.mix(i, i+1)
        
    # --- VARIATIONAL LAYER 2 ---
    for i in range(num_features):
        pc.shift(i, weights_block[i + 12])
        
    # Introduce global interference / "Entanglement"
    pc.dft()
    
    return pc

# Visualize
dummy_w = torch.zeros(24)
test_pc = create_vpc_block(X[0], dummy_w)
print("VPC Block Preview:")
pf.draw(test_pc, mode='text')

VPC Block Preview:
T0: ──[S(5.54)]───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────[S(0.00)]─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────[S(0.00)]─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────┤
                                                                                                                                                                                                                                                                              [MIX]                                                                                                                                                                           │    
T1: ─────────────[S(5.75)]──────────────────────────────────────────────────

## 3. Deep Predictor & Loss Function
The multi-layer pipeline feeds sequentially through the models. We will define a 2-Block stack (48 parameters) to analyze extreme feature correlation. We do NOT use non-linear thresholding between layers, just let the phases cascade.

In [4]:
engine = AnalyticEngine()

def stacked_vpc_predict(x, all_weights, num_blocks=2):
    current_phases = x
    
    for b in range(num_blocks):
        # Extract the 24 parameters strictly for this block
        block_w = all_weights[b*24 : (b+1)*24]
        
        # Build and run the single block
        pc = create_vpc_block(current_phases, block_w, block_id=b)
        res = engine.run(pc)
        
        # Directly pass the intermediate pure phases to the next layer
        current_phases = res['phases']
            
    # Read the phase of Thread 0 for classification logic
    phase_0 = current_phases[0]
    
    # Map phase [-pi, pi] to probability [0, 1] using (sin(theta) + 1) / 2
    probability = (torch.sin(phase_0) + 1.0) / 2.0
    return probability
    
def calculate_stacked_loss(all_weights, num_blocks=2):
    predictions = torch.stack([stacked_vpc_predict(x, all_weights, num_blocks) for x in X])
    # Mean Squared Error
    mse = torch.mean((predictions - y)**2)
    return mse

NUM_BLOCKS = 2
TOTAL_PARAMS = 24 * NUM_BLOCKS

# Initial loss with random weights
initial_weights = torch.empty(TOTAL_PARAMS).uniform_(-math.pi, math.pi)
initial_weights.requires_grad_(True)
initial_loss = calculate_stacked_loss(initial_weights, num_blocks=NUM_BLOCKS)
print(f"Initial Random MSE Loss (Stacked): {initial_loss.item():.4f}")

Initial Random MSE Loss (Stacked): 0.3002


## 4. Deep Optimization (COBYLA)
We utilize `scipy.optimize.minimize` (COBYLA) over the full 48-variable state space to find a converging solution tracking continuous complex topological distributions.

In [5]:
print("Starting Deep Optimization natively across Phasor bounds... (Adam)")

optimizer = optim.Adam([initial_weights], lr=0.1)

for epoch in range(100):
    optimizer.zero_grad()
    loss = calculate_stacked_loss(initial_weights, num_blocks=NUM_BLOCKS)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/100 | Loss: {loss.item():.4f}")

optimal_weights = initial_weights.detach()
final_loss = loss.item()

print(f"Optimization Complete!")
print(f"Final Trained MSE Loss: {final_loss:.4f}")

Starting Deep Optimization natively across Phasor bounds... (Adam)


Epoch 20/100 | Loss: 0.0781


Epoch 40/100 | Loss: 0.0508


Epoch 60/100 | Loss: 0.0507


Epoch 80/100 | Loss: 0.0589


Epoch 100/100 | Loss: 0.0596
Optimization Complete!
Final Trained MSE Loss: 0.0596


## 5. Evaluation
Verifying the classification capacity of the multi-layer stack without non-linear gates.

In [6]:
final_probs = torch.stack([stacked_vpc_predict(x, optimal_weights, NUM_BLOCKS) for x in X])
binary_predictions = (final_probs > 0.5).float()

accuracy = torch.mean((binary_predictions == y).float()) * 100
print(f"Final Classification Accuracy: {accuracy.item():.2f}%")

Final Classification Accuracy: 92.00%
